# GeFL Class-Balanced — Full paper-style sweep (Kaggle 2×T4, headless)

**Execution:** Save Version → Save & Run All. Runs headless up to 12h.

**Purpose:** paper-quality sweep. Configure top cell with any subset of {datasets × generators × mechanisms × IFs × α × seeds} and fire. Notebook expands cartesian product and runs each combo. **Idempotent** — completed runs are skipped on rerun.

**Rough per-run cost** (100+100 rounds, T4 wall):

| generator | MNIST/FMNIST | CIFAR-10 | CIFAR-100 |
|---|---|---|---|
| vae | ~5-10 min | ~50 min | ~60 min |
| gan | ~5-10 min | ~50-60 min | ~60-80 min |
| ddpm | ~20-40 min | ~3-4 h | ~3-4 h |

Two runs go in parallel (one per T4), so wall time ~ (runs / 2) × per-run. If total exceeds 12h, set `MAX_RUNS_THIS_SESSION` to cap this pass, then rerun the notebook in a fresh Kaggle session to continue.

**Results land in** `logs/{dataset}/{gen}/{tag}.csv`. Bundle zipped at end for one-click download.

In [ ]:
# ============= SET THESE BEFORE Save & Run All =============
# Pick any subset of each list. Notebook expands into the cartesian
# product and runs every combo. Idempotent — completed runs are skipped
# on rerun (checked by output CSV filename).

FRAMEWORKS = ['gefl']                                 # 'gefl' | 'gefl_f' (feature-space variant)
DATASETS   = ['mnist']                                # 'mnist' | 'fmnist' | 'cifar10' | 'cifar100'
GENERATORS = ['vae']                                  # 'vae' | 'gan' | 'ddpm'
MECHANISMS = ['baseline', 'a_only', 'b_only', 'proposed']  # any subset
IMBALANCE_FACTORS = [0.01, 0.1, 1.0]
DIR_PARAMS        = [0.3]
SEEDS             = [0, 1, 2]

# Federated schedule (paper Table XIV defaults)
GEN_WU_EPOCHS = 100    # T_KA / 2
EPOCHS        = 100    # T_TN
SAMPLE_TEST   = 5

# Optional: cap total runs to fit a 12h Kaggle session.
MAX_RUNS_THIS_SESSION = 0    # 0 = no cap

# =========== REPO ============
REPO_URL = 'https://github.com/Raunak4518/Fedlearning.git'
BRANCH   = 'geflf-improvement'   # branch you want to run
# ============================================================

# framework → dataset → config file
CONFIG_MAP = {
    'gefl': {
        'mnist':    'configs/mnist_lt.yaml',
        'fmnist':   'configs/fmnist_lt.yaml',
        'cifar10':  'configs/cifar10_lt.yaml',
        'cifar100': 'configs/cifar100_lt.yaml',
    },
    'gefl_f': {
        'mnist':    'configs/mnist_gefl_f.yaml',
        'fmnist':   'configs/fmnist_gefl_f.yaml',
        'cifar10':  'configs/cifar10_gefl_f.yaml',
        'cifar100': 'configs/cifar100_gefl_f.yaml',
    },
}
GEN_HINT = {'vae': ' (fast, ~1x)', 'gan': ' (~1-1.2x VAE)', 'ddpm': ' (~3-5x VAE, expensive)'}

for fw in FRAMEWORKS:
    assert fw in CONFIG_MAP, f'FRAMEWORKS entry {fw!r} not in {list(CONFIG_MAP)}'
    for d in DATASETS:
        assert d in CONFIG_MAP[fw], f'no {fw} config for dataset {d!r}'
for g in GENERATORS:
    assert g in ('vae','gan','ddpm'), f'GENERATORS entry {g!r} invalid'
for m in MECHANISMS:
    assert m in ('baseline','a_only','b_only','proposed'), f'MECHANISMS entry {m!r} invalid'

total_runs = (len(FRAMEWORKS) * len(DATASETS) * len(GENERATORS) * len(MECHANISMS)
              * len(IMBALANCE_FACTORS) * len(DIR_PARAMS) * len(SEEDS))
print(f'Frameworks : {FRAMEWORKS}')
print(f'Datasets   : {DATASETS}')
print(f'Generators : {[g + GEN_HINT[g] for g in GENERATORS]}')
print(f'Mechanisms : {MECHANISMS}')
print(f'IFs        : {IMBALANCE_FACTORS}')
print(f'α          : {DIR_PARAMS}')
print(f'Seeds      : {SEEDS}')
print(f'-> {total_runs} runs total')
if MAX_RUNS_THIS_SESSION > 0:
    print(f'-> cap this session at {MAX_RUNS_THIS_SESSION} runs')

In [ ]:
# ----- Clone the code from GitHub -----
import os, subprocess, sys

PROJECT_ROOT = '/kaggle/working/project'
if os.path.exists(PROJECT_ROOT):
    subprocess.check_call(['rm', '-rf', PROJECT_ROOT])

clone_url = REPO_URL
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret('GITHUB_TOKEN')
    if token and REPO_URL.startswith('https://github.com/'):
        clone_url = REPO_URL.replace('https://', f'https://x-access-token:{token}@')
    print('Using GITHUB_TOKEN secret (private-repo path).')
except Exception:
    print('No GITHUB_TOKEN secret set — assuming public repo.')

subprocess.check_call(['git', 'clone', '--branch', BRANCH, '--depth', '1', clone_url, PROJECT_ROOT])
os.chdir(PROJECT_ROOT)
sys.path.insert(0, PROJECT_ROOT)

commit = subprocess.check_output(['git', '-C', PROJECT_ROOT, 'rev-parse', 'HEAD'], text=True).strip()
print(f'\nCloned {BRANCH} @ {commit[:12]}')

In [ ]:
# ----- Install requirements (skip torch: Kaggle ships GPU-matched build) -----
req = os.path.join(PROJECT_ROOT, 'requirements.txt')
if os.path.exists(req):
    filtered = '/kaggle/working/requirements_no_torch.txt'
    with open(req) as f, open(filtered, 'w') as g:
        for line in f:
            if not line.strip().lower().startswith(('torch', 'torchvision')):
                g.write(line)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', filtered])
print('Deps installed.')

In [ ]:
# ----- GPU sanity: expect 2 T4s -----
import torch
n = torch.cuda.device_count()
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '| devices:', n)
for i in range(n):
    print(f'  cuda:{i} =', torch.cuda.get_device_name(i))
assert n >= 2, 'Enable Settings → Accelerator → GPU T4 x2.'

In [ ]:
# ----- Build the (config -> cmd) plan, then run pairs in parallel on 2 GPUs -----
import os, sys, subprocess, time, shlex, itertools
import pandas as pd

os.makedirs('./logs', exist_ok=True)
MECH_FLAGS = {'baseline': (0, 0), 'a_only': (1, 0), 'b_only': (0, 1), 'proposed': (1, 1)}

# Per-generator hyperparameter overrides. Config files assume gen_model=vae;
# GAN needs different Adam betas + lr; DDPM has its own lr + weight decay + timesteps.
GEN_OVERRIDES = {
    'vae':  ['--gen_lr', '0.001'],
    'gan':  ['--gen_lr', '0.0002', '--b1', '0.5', '--b2', '0.999'],
    'ddpm': ['--gen_lr', '0.0001', '--weight_decay_ddpm', '0.001', '--n_T', '200'],
}

def _run_cmd(framework, dataset, gen, mech, imb, dir_p, seed, out_csv):
    cmd = [
        sys.executable, 'scripts/sweep.py',
        '--framework', framework,
        '--config', CONFIG_MAP[framework][dataset],
        '--imbalance_factors', str(imb),
        '--dir_params', str(dir_p),
        '--seeds', str(seed),
        '--mechanisms', mech,
        '--out_csv', out_csv,
        '--gen_model', gen,
        '--gen_wu_epochs', str(GEN_WU_EPOCHS),
        '--epochs', str(EPOCHS),
        '--sample_test', str(SAMPLE_TEST),
        '--eval_centralized_upper_bound', '0',
    ] + GEN_OVERRIDES.get(gen, [])
    return cmd

# Expand cartesian product; one CSV per run for idempotence.
plan = []
for framework, dataset, gen, mech, imb, dir_p, seed in itertools.product(
        FRAMEWORKS, DATASETS, GENERATORS, MECHANISMS, IMBALANCE_FACTORS, DIR_PARAMS, SEEDS):
    log_dir = f'./logs/{framework}/{dataset}/{gen}'
    os.makedirs(log_dir, exist_ok=True)
    tag = f'{mech}_imb{imb}_dir{dir_p}_seed{seed}'.replace('.', 'p')
    out_csv = f'{log_dir}/{tag}.csv'
    plan.append({
        'framework': framework, 'dataset': dataset, 'gen': gen, 'mech': mech,
        'imb': imb, 'dir': dir_p, 'seed': seed,
        'tag': f'{framework}/{dataset}/{gen}/{tag}',
        'csv': out_csv,
        'cmd': _run_cmd(framework, dataset, gen, mech, imb, dir_p, seed, out_csv),
    })

todo = [p for p in plan if not os.path.exists(p['csv'])]
print(f'Plan: {len(plan)} total; {len(plan)-len(todo)} already done; {len(todo)} to run.')

if MAX_RUNS_THIS_SESSION and len(todo) > MAX_RUNS_THIS_SESSION:
    todo = todo[:MAX_RUNS_THIS_SESSION]
    print(f'Capping to {len(todo)} this session (rerun notebook to continue).')

# Chunk into pairs (2 concurrent GPUs).
pairs = [todo[i:i+2] for i in range(0, len(todo), 2)]
print(f'{len(pairs)} pairs to fire.')

def _run_pair(pair_idx, entries):
    procs, logs = [], []
    for gpu, e in enumerate(entries):
        env = os.environ.copy(); env['CUDA_VISIBLE_DEVICES'] = str(gpu)
        log_path = f'/kaggle/working/pair{pair_idx:04d}_gpu{gpu}.log'
        lf = open(log_path, 'a')
        lf.write(f'\n\n==== {e["tag"]}\n==== {" ".join(map(shlex.quote, e["cmd"]))} ====\n'); lf.flush()
        p = subprocess.Popen(e['cmd'], env=env, stdout=lf, stderr=subprocess.STDOUT)
        procs.append(p); logs.append(lf)
        print(f'  cuda:{gpu}  {e["tag"]}  pid={p.pid}')
    while any(p.poll() is None for p in procs):
        time.sleep(600)
        state = ', '.join(f'gpu{i}:{"done" if p.poll() is not None else "running"}' for i, p in enumerate(procs))
        print(f'    {time.strftime("%H:%M:%S")}  {state}')
    for lf in logs: lf.close()
    for i, p in enumerate(procs):
        assert p.returncode == 0, f'gpu{i} failed rc={p.returncode} — inspect its log'
    for e in entries:
        try:
            d = pd.read_csv(e['csv']).iloc[-1]
            print(f'    {e["tag"]}  overall={d.get("acc_overall",0):.3f}  '
                  f'head={d.get("acc_head",0):.3f}  tail={d.get("acc_tail",0):.3f}  '
                  f'genLA={d.get("gen_label_accuracy", float("nan"))}')
        except Exception as ex:
            print(f'    {e["tag"]}  (peek failed: {ex})')

for i, entries in enumerate(pairs):
    print(f'\n=== Pair {i+1}/{len(pairs)} ===')
    t0 = time.time()
    _run_pair(i, entries)
    print(f'  pair wall: {(time.time()-t0)/60:.1f} min')

In [ ]:
# ----- Aggregate all per-run CSVs across frameworks + datasets + generators -----
import glob, pandas as pd, os, re

def _parse_from_path(path):
    # ./logs/{framework}/{dataset}/{gen}/{mech}_imb0p01_dir0p3_seed0.csv
    parts = path.replace('\\', '/').split('/')
    if len(parts) < 5: return None
    framework, dataset, gen, fname = parts[-4], parts[-3], parts[-2], parts[-1]
    base = os.path.splitext(fname)[0]
    m = re.match(r'(?P<mech>[a-z_]+?)_imb(?P<imb>[0-9p]+)_dir(?P<dir>[0-9p]+)_seed(?P<seed>[0-9]+)', base)
    if not m: return None
    d = m.groupdict()
    d['imb']  = float(d['imb'].replace('p', '.'))
    d['dir']  = float(d['dir'].replace('p', '.'))
    d['seed'] = int(d['seed'])
    d['framework'] = framework
    d['dataset']   = dataset
    d['gen']       = gen
    return d

frames = []
for path in sorted(glob.glob('./logs/*/*/*/*.csv')):
    tag = _parse_from_path(path)
    if tag is None: continue
    try:
        d = pd.read_csv(path)
        for k, v in tag.items(): d[k] = v
        frames.append(d)
    except Exception as e:
        print(f'skip {path}: {e}')

if not frames:
    print('NO RESULTS. Sweep produced no CSVs.')
else:
    df = pd.concat(frames, ignore_index=True)
    wanted = ['framework', 'dataset', 'gen', 'mech', 'imb', 'dir', 'seed',
              'acc_overall', 'acc_head', 'acc_medium', 'acc_tail',
              'macro_f1', 'class_balanced_accuracy',
              'gen_label_accuracy', 'gen_mean_confidence', 'mnd_ratio']
    cols = [c for c in wanted if c in df.columns]
    print('=== Per-run results ===')
    print(df[cols].sort_values(['framework','dataset','gen','mech','imb','seed']).to_string(index=False))

    numeric = [c for c in cols if c not in ('framework','dataset','gen','mech','imb','dir','seed')]
    agg = df.groupby(['framework','dataset','gen','mech','imb','dir'])[numeric].agg(['mean', 'std']).round(4)
    print('\n=== Mean ± std across seeds ===')
    print(agg.to_string())

    df.to_csv('./logs/_merged.csv', index=False)
    print('\nMerged: ./logs/_merged.csv')

In [ ]:
# ----- Bundle results for download -----
import shutil
bundle_dir = '/kaggle/working/gefl_sweep_bundle'
if os.path.exists(bundle_dir):
    shutil.rmtree(bundle_dir)
os.makedirs(bundle_dir)
if os.path.exists('./logs'):
    shutil.copytree('./logs', os.path.join(bundle_dir, 'logs'))
shutil.make_archive('/kaggle/working/gefl_sweep_results', 'zip', bundle_dir)
print('Bundle: /kaggle/working/gefl_sweep_results.zip')